In [8]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import transforms,datasets
from torchvision.transforms import Compose

In [22]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_transform=Compose([transforms.RandomRotation(degrees=(-30,30)),transforms.ToTensor()])
test_transform=Compose([transforms.ToTensor()])
train_dataset=datasets.MNIST(train=True,root="data",download=True,transform=train_transform)
test_dataset=datasets.MNIST(train=False,root="data",download=True,transform=test_transform)
train_loader=DataLoader(dataset=train_dataset,batch_size=64,shuffle=True)
test_loader=DataLoader(dataset=test_dataset,batch_size=64,shuffle=False)
print(next(iter(train_loader))[0].shape)

torch.Size([64, 1, 28, 28])


In [17]:
class RNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn=nn.RNN(input_size=28,hidden_size=64,num_layers=5,batch_first=True)
        self.fc=nn.Linear(64,10)
    def forward(self,x):
        rnn_output,hidden_state=self.rnn(x)
        x=rnn_output[:,-1,:]
        predicted_output=self.fc(x)
        return predicted_output


In [30]:
epoch=5
model=RNN()
model.train()
model=model.to(device)
optimizer=torch.optim.Adam(model.parameters(),lr=0.001)
loss_fn=nn.CrossEntropyLoss()
for i in range(epoch):
    model.train()
    train_loss=0
    for data,result in train_loader:
        data=data.squeeze(1)
        data=data.to(device)
        result=result.to(device)
        optimizer.zero_grad()
        predicted_result=model(data)
        batch_loss=loss_fn(predicted_result,result)
        batch_loss.backward()
        optimizer.step()
        train_loss+=batch_loss.item()
    model.eval()
    validation_loss=0
    for data,result in test_loader:
        data=data.squeeze(1)
        data=data.to(device)
        result=result.to(device)
        validation_result=model(data)
        validation_loss+=loss_fn(validation_result,result)
    print(f"Epoch {i} Training Loss {train_loss/len(train_loader)} Validation Loss {validation_loss/len(test_loader)}")

Epoch 0 Training Loss 0.8550761575892027 Validation Loss 0.35306400060653687
Epoch 1 Training Loss 0.3750329350850094 Validation Loss 0.17632566392421722
Epoch 2 Training Loss 0.27909152159320394 Validation Loss 0.14842279255390167
Epoch 3 Training Loss 0.23428720664749267 Validation Loss 0.16151094436645508
Epoch 4 Training Loss 0.2030948510012234 Validation Loss 0.12327849119901657


In [45]:
correct=0
total=0
for data,result in test_loader:
    data=data.squeeze(1)
    data=data.to(device)
    result=result.to(device)
    predicted_result=model(data)
    correct+=(torch.argmax(predicted_result,dim=1)==result).sum().item()
    total+=result.size(0)
print(f"Accuracy : {correct/total*100}%")

Accuracy : 96.39%
